In [2]:
import os
import time
import mne
import numpy as np
import pandas as pd

import bsl
from bsl import StreamPlayer, datasets
# from bsl.externals import pylsl  # distributed version of pylsl
from bsl.triggers import TriggerDef

import pylsl

import pickle

import math
import matplotlib
import matplotlib.pyplot as plt
from pythonosc.udp_client import SimpleUDPClient

In [5]:
from pylsl import resolve_streams

streams = resolve_streams()
for s in streams:
    print(s.name(), s.type(), s.channel_count(), s.nominal_srate())

OxySoft NIRS 28 75.0
OxySoft Event Marker Markers 1 0.0


### OSC Client Intialization

In [ ]:
# OSC client initialization
ip = "127.0.0.1"
port = 7000
client = SimpleUDPClient(ip, port)

### Load Pretrained Model

In [ ]:
# Load your pretrained model and scaler
with open("model.pkl", "rb") as f_m:
    model = pickle.load(f_m)
with open("scaler.pkl", "rb") as f_s:
    scaler = pickle.load(f_s)
with open("pca.pkl", "rb") as f_p:
    pca = pickle.load(f_p)

## Analyse Signal

In [ ]:
from pylsl import resolve_streams, StreamInlet
from collections import deque
import numpy as np
import mne
import time

# ── Constants ────────────────────────────────────────────────────────────────
FNIRS_FS = 75.0
WINDOW_S = 10
FNIRS_WINDOW_SAMPLES = int(FNIRS_FS * WINDOW_S)  # 750 samples
UPDATE_EVERY_S = 0.5
N_FNIRS_CHANNELS = 28

# ── MNE info objects ─────────────────────────────────────────────────────────

ch_names = [
    'Rx1 - Tx1 O2Hb', 'Rx1 - Tx1 HHb',
    'Rx1 - Tx3 O2Hb', 'Rx1 - Tx3 HHb',
    'Rx2 - Tx1 O2Hb', 'Rx2 - Tx1 HHb',
    'Rx2 - Tx3 O2Hb', 'Rx2 - Tx3 HHb',
    'Rx3 - Tx4 O2Hb', 'Rx3 - Tx4 HHb',
    'Rx3 - Tx5 O2Hb', 'Rx3 - Tx5 HHb',
    'Rx8 - Tx9 O2Hb', 'Rx8 - Tx9 HHb',
    'Rx8 - Tx10 O2Hb', 'Rx8 - Tx10 HHb',
    'Rx5 - Tx6 O2Hb', 'Rx5 - Tx6 HHb',
    'Rx5 - Tx8 O2Hb', 'Rx5 - Tx8 HHb',
    'Rx6 - Tx6 O2Hb', 'Rx6 - Tx6 HHb',
    'Rx6 - Tx8 O2Hb', 'Rx6 - Tx8 HHb',
    'Rx4 - Tx2 O2Hb', 'Rx4 - Tx2 HHb',
    'Rx7 - Tx7 O2Hb', 'Rx7 - Tx7 HHb',
]

fnirs_info = mne.create_info(
    ch_names=ch_names,
    sfreq=FNIRS_FS,
    ch_types=['fnirs_cw_amplitude'] * 28
)

# ── LSL inlets ───────────────────────────────────────────────────────────────

streams = resolve_streams()

fnirs_stream = [s for s in streams if s.name() == 'OxySoft'][0]
fnirs_inlet = StreamInlet(fnirs_stream)

# ── Ring buffer ──────────────────────────────────────────────────────────────

fnirs_buffer = deque(maxlen=FNIRS_WINDOW_SAMPLES)

# ── Main loop ────────────────────────────────────────────────────────────────

last_process_time = time.time()

while True:

    # ── Ingest fNIRS stream ────────────────────────────────────────────────────

    fnirs_samples, _ = fnirs_inlet.pull_chunk(timeout=0.0)

    if fnirs_samples:
        fnirs_buffer.extend(fnirs_samples)

    # ── Wait for fNIRS buffer to fill ─────────────────────────────────────────

    if len(fnirs_buffer) < FNIRS_WINDOW_SAMPLES:
        print(
            f"Buffering... fNIRS "
            f"{len(fnirs_buffer)}/{FNIRS_WINDOW_SAMPLES}"
        )
        continue

    # ── Rate limit processing ─────────────────────────────────────────────────

    now = time.time()

    if now - last_process_time < UPDATE_EVERY_S:
        continue

    last_process_time = now

    # ── fNIRS data ────────────────────────────────────────────────────────────

    fnirs_data = np.array(fnirs_buffer)  # (750, 28)
    fnirs_data = np.nan_to_num(fnirs_data)

    # ── Calculate features ───────────────────────────────────────────────────
    #
    # The scaler expects 112 features in this exact order:
    #
    # Channel 1 O2Hb: min, max, mean, std
    # Channel 1 HHb:  min, max, mean, std
    # Channel 2 O2Hb: min, max, mean, std
    # Channel 2 HHb:  min, max, mean, std
    # ...
    #
    # 28 channels × 4 features = 112 features

    features = []

    for channel_idx in range(N_FNIRS_CHANNELS):

        channel_data = fnirs_data[:, channel_idx]

        channel_min = np.min(channel_data)
        channel_max = np.max(channel_data)
        channel_mean = np.mean(channel_data)
        channel_std = np.std(channel_data)

        features.extend([
            channel_min,
            channel_max,
            channel_mean,
            channel_std
        ])

    features = np.array(features)

    # ── Verify feature count ─────────────────────────────────────────────────

    print("Number of features:", len(features))
    print("Scaler expects:", scaler.n_features_in_)

    # ── Scale & prediction ───────────────────────────────────────────────────

    # 112 raw features
    features = np.array(features).reshape(1, -1)

    # Scale
    features_scaled = scaler.transform(features)

    # PCA
    features_pca = pca.transform(features_scaled)

    # Predict
    prediction = model.predict(features_pca)

    print("Prediction:", prediction)

## Send Signal Via Osc

In [ ]:

client.send_message("/in/prediction", float(prediction[0]))

## open code code

Key design decisions:
- Window alignment: Both use the same 10s window, so features are temporally aligned
- Feature vector: [alpha_power, hbo_mean, hbr_mean] — you'd likely expand this (multiple EEG frequency bands, HbO/HbR slope, etc.)
- Scaler: fNIRS units (µM) differ wildly from EEG alpha power, so StandardScaler fit offline is essential
- Rate: The combined loop runs at whatever rate you set (4 Hz like the original, or slower to match fNIRS)

In [ ]:
# Main loop
while True:
    receiver.acquire()

    # --- EEG stream (disabled) ---
    # eeg_data, _ = receiver.get_window(stream_name='igeb')
    # eeg_data = np.nan_to_num(eeg_data)
    # eeg_raw = mne.io.RawArray(data=eeg_data[:, [False, True]].T, info=eeg_info)
    # eeg_raw.filter(1, 30)
    # eeg_raw.crop(tmin=9)
    # psds, _ = mne.time_frequency.psd_welch(eeg_raw, fmin=8, fmax=12, n_fft=125)
    # alpha_score = np.mean(psds)
    # alpha_norm = (alpha_score - ref_mean_score) / ref_std_score / 2

    # --- fNIRS stream ---
    fnirs_data, _ = receiver.get_window(stream_name='fnirs_stream_name')
    fnirs_data = np.nan_to_num(fnirs_data)
    # Assuming channel 0 = timestamp, 1 = HbO, 2 = HbR
    fnirs_raw = mne.io.RawArray(data=fnirs_data[:, [False, True, True]].T,
                                 info=fnirs_info)
    fnirs_raw.filter(0.01, 0.5)  # Hemodynamic response is slow
    fnirs_raw.crop(tmin=9)
    # Convert to concentration (requires MNE fnirs processing)
    # fnirs_conc = mne.preprocessing.nirs.beer_lambert_law(fnirs_raw)
    # Or just use raw amplitude features
    hbo_mean = fnirs_raw.get_data(picks=['HbO']).mean()
    hbr_mean = fnirs_raw.get_data(picks=['HbR']).mean()

    # --- Combine and predict ---
    features = np.array([[hbo_mean, hbr_mean]])  # EEG (alpha_norm) disabled
    features_scaled = scaler.transform(features)
    prediction = model.predict(features_scaled)

    # Send to VR/neuromore
    client.send_message("/in/prediction", float(prediction[0]))